# Experiment 2: Comparative Analysis of the baseline models and the 8-bit Quantized Whisper Version

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from jiwer import wer, cer
from datasets import load_dataset
import re
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import jiwer

## Whisper Baseline vs. 8-bit Quantized Whisper
First I will check the distributions of the logged CPU and Wall times from both test runs to decide whether to perform a Paired T-Test or a Wilcoxon Signed Rank Test. 

In [ ]:
def load_nested_jsonl_to_df(file_path: str) -> pd.DataFrame:
    flattened_records = []
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue  # Skip empty lines
                
            try:
                if line_number <= 1:
                    continue
                # Parse the JSON string into a Python object (expected to be a list)
                line_data = json.loads(line)
                
                # Check if the parsed object is actually a list
                if isinstance(line_data, list):
                    # .extend() un-nests the list and adds the individual dicts
                    flattened_records.extend(line_data)
                else:
                    print(f"Warning: Line {line_number} is not a list. Skipping.")
                    
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON on line {line_number}: {e}")
                
    # Convert the flat list of dictionaries into a DataFrame
    return pd.DataFrame(flattened_records)

In [ ]:
def load_logs_to_df(file_path: str) -> pd.DataFrame:
    flattened_records = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                if line_number <= 12:
                    continue    # Skipping the first lines representing the first 5 inference runs

                match = re.search(r"Walltime:\s*([0-9]+(?:\.[0-9]+)?)\s+CPU time:\s*([0-9]+(?:\.[0-9]+)?)", line)
                if not match:
                    continue

                walltime = float(match.group(1))
                cputime = float(match.group(2))

                flattened_records.append({
                    'walltime': walltime,
                    'cputime': cputime
                })
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON on line {line_number}: {e}")
                
    # Convert the flat list of dictionaries into a DataFrame
    return pd.DataFrame(flattened_records)

In [ ]:
efficiency_int8_df = load_logs_to_df(file_path='../results/whisper_int8.log')
efficiency_baseline_df = load_logs_to_df(file_path='../results/whisper_baseline.log')
print(efficiency_baseline_df.shape)
print(efficiency_int8_df.shape)

whisper_baseline_df = load_nested_jsonl_to_df('../results/whisper_baseline.jsonl')
whisper_int8_df = load_nested_jsonl_to_df('../results/whisper_int8.jsonl')
print(whisper_baseline_df.shape)
print(whisper_int8_df.shape)

# Combine the dfs
whisper_b_merged_df = efficiency_baseline_df.join(whisper_baseline_df)
whisper_q_merged_df = efficiency_int8_df.join(whisper_int8_df)

In [ ]:
import json
import re
import pandas as pd

def log_df(log_file: str, json_file: str):
    # Extract timings
    timings = []
    pattern = re.compile(
        r"Walltime:\s*([0-9]+(?:\.[0-9]+)?)\s+CPU time:\s*([0-9]+(?:\.[0-9]+)?)"
    )

    with open(log_file) as f:
        for line in f:
            m = pattern.search(line)
            if m:
                timings.append({
                    "walltime": float(m.group(1)),
                    "cputime": float(m.group(2))
                })

    # Read batches
    batches = []
    with open(json_file) as f:
        for line in f:
            batches.append(json.loads(line))   # each line is a list of dicts

    assert len(timings) == len(batches)

    df = pd.DataFrame({
        "batch": range(len(batches)),
        "samples": batches,
        "walltime": [t["walltime"] for t in timings],
        "cputime": [t["cputime"] for t in timings],
    })

    return df

In [ ]:
baseline_df = log_df(log_file='../results/whisper_baseline.log', json_file='../results/whisper_baseline.jsonl')
int8_df = log_df(log_file='../results/whisper_int8.log', json_file='../results/whisper_int8.jsonl')

In [ ]:
import pandas as pd
def unpack(df: pd.DataFrame):
    # 1. Compute the total duration for the list of dicts and assign to a new column
    df['duration'] = df['samples'].apply(lambda lst: sum(d['seg_end'] - d['seg_start'] for d in lst))

    # 2. Drop the original 'sample' column
    df = df.drop(columns=['samples'])
    return df

In [ ]:
baseline_df = unpack(baseline_df)
int8_df = unpack(int8_df)

baseline_df['rtf'] = baseline_df['walltime'] / baseline_df['duration']
int8_df['rtf'] = int8_df['walltime'] / int8_df['duration']

baseline_df['tps'] = 1 / baseline_df['rtf']
int8_df['tps'] = 1 / int8_df['rtf']

int8_df = int8_df.iloc[:len(baseline_df)]

In [ ]:
print(baseline_df.shape)
print(int8_df.shape)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_simple_histograms(df: pd.DataFrame, dataset_name: str):
    # Apply Seaborn styling globally
    sns.set_theme(style="whitegrid")

    # Create a simple 1x2 grid for RTF and TPS
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Efficiency Distributions: {dataset_name}', fontsize=14, fontweight='bold')

    # Plot RTF
    sns.histplot(data=df, x='rtf', bins=30, ax=axes[0], edgecolor='black')
    axes[0].set_title('RTF Distribution')
    axes[0].set_xlabel('Real-Time Factor (RTF)')
    axes[0].set_ylabel('Frequency')

    # Plot TPS
    sns.histplot(data=df, x='tps', bins=30, ax=axes[1], edgecolor='black')
    axes[1].set_title('TPS Distribution')
    axes[1].set_xlabel('Throughput Per Second (TPS)')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_simple_histograms(baseline_df, 'Baseline')

In [ ]:
plot_simple_histograms(int8_df, '8-bit Quantized')

In [ ]:
import numpy as np
from scipy import stats

def wilcoxon_t_test(df_int8: pd.DataFrame, df_baseline: pd.DataFrame):
    rtf_int8 = df_int8.loc[:, 'rtf']
    rtf_baseline = df_baseline.loc[:, 'rtf']

    tps_int8 = df_int8.loc[:, 'tps']
    tps_baseline = df_baseline.loc[:, 'tps']

    rtf_stat, rtf_p_value = stats.wilcoxon(rtf_baseline, rtf_int8)

    # For TPS
    tps_stat, tps_p_value = stats.wilcoxon(tps_baseline, tps_int8)

    # ---------------------------------------------------------
    # 3. Print and interpret the results
    # ---------------------------------------------------------
    alpha = 0.05  # Standard significance level

    print("=== Real-Time Factor (RTF) Comparison ===")
    print(f"Test Statistic: {rtf_stat:.4f}, p-value: {rtf_p_value:.4f}")
    if rtf_p_value < alpha:
        print("Result: Significant difference in RTF between the two models.\n")
    else:
        print("Result: No significant difference in RTF.\n")

    print("=== Throughput Per Second (TPS) Comparison ===")
    print(f"Test Statistic: {tps_stat:.4f}, p-value: {tps_p_value:.4f}")
    if tps_p_value < alpha:
        print("Result: Significant difference in TPS between the two models.")
    else:
        print("Result: No significant difference in TPS.")

In [ ]:
wilcoxon_t_test(df_int8=int8_df, df_baseline=baseline_df)

## A Wilcoxon Signed Rank Test
This is to compare the ASR quality between the full precision and quantized whisper model on only the clean CoRal data, since full precision Whisper was not able to run on the hospital server, making it impossible to test it on the Wrist-Angel data.

In [ ]:
def load_nested_jsonl_to_df(file_path: str) -> pd.DataFrame:
    flattened_records = []
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue  # Skip empty lines
                
            try:
                # Parse the JSON string into a Python object (expected to be a list)
                line_data = json.loads(line)
                
                # Check if the parsed object is actually a list
                if isinstance(line_data, list):
                    # .extend() un-nests the list and adds the individual dicts
                    flattened_records.extend(line_data)
                else:
                    print(f"Warning: Line {line_number} is not a list. Skipping.")
                    
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON on line {line_number}: {e}")
                
    # Convert the flat list of dictionaries into a DataFrame
    return pd.DataFrame(flattened_records)

In [ ]:
def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Calculate WER and CER per row (utterance)
    df['wer'] = df.apply(lambda row: jiwer.wer(row['ref'], row['hyp']), axis=1)
    df['cer'] = df.apply(lambda row: jiwer.cer(row['ref'], row['hyp']), axis=1)
    
    return df

In [ ]:
coral_baseline_df = load_nested_jsonl_to_df('/thesis_multi_speaker_asr/results/whisper_baseline.jsonl')
processed_coral_baseline_df = process_dataset(coral_baseline_df)

coral_int8_df = load_nested_jsonl_to_df('/thesis_multi_speaker_asr/results/whisper_int8.jsonl')
processed_coral_int8_df = process_dataset(coral_int8_df)

In [ ]:
print(processed_coral_int8_df.shape)
print(processed_coral_baseline_df.shape)

processed_coral_int8_df = processed_coral_int8_df.iloc[:len(processed_coral_baseline_df)]

print(processed_coral_int8_df.shape)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_histograms(df: pd.DataFrame, model_name: str):
    # Apply Seaborn styling globally
    sns.set_theme(style="whitegrid")

    # Create a simple 1x2 grid for RTF and TPS
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Error Rate Distribution: {model_name}', fontsize=14, fontweight='bold')

    custom_bins = np.arange(0, 5.05, 0.05)
    # Plot RTF
    sns.histplot(data=df, x='wer', bins=custom_bins, ax=axes[0], edgecolor='black')
    axes[0].set_title('WER Distribution')
    axes[0].set_xlabel('Word Error Rate (WER)')
    axes[0].set_ylabel('Count')
    axes[0].set_yscale('log')

    # Plot TPS
    sns.histplot(data=df, x='cer', bins=custom_bins, ax=axes[1], edgecolor='black')
    axes[1].set_title('CER Distribution')
    axes[1].set_xlabel('Character Error Rate (CER)')
    axes[1].set_ylabel('Count')
    axes[1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms(processed_coral_baseline_df, model_name='Whisper Baseline')

In [ ]:
plot_histograms(processed_coral_int8_df, model_name='Whisper 8-bit Quantized')

In [ ]:
import numpy as np
from scipy import stats

def wilcoxon_t_test(df_int8: pd.DataFrame, df_baseline: pd.DataFrame):
    wer_int8 = df_int8.loc[:, 'wer']
    wer_baseline = df_baseline.loc[:, 'wer']

    cer_int8 = df_int8.loc[:, 'cer']
    cer_baseline = df_baseline.loc[:, 'cer']

    wer_stat, wer_p_value = stats.wilcoxon(wer_int8, wer_baseline, alternative='less')

    # For TPS
    cer_stat, cer_p_value = stats.wilcoxon(cer_int8, cer_baseline, alternative='less')

    # ---------------------------------------------------------
    # 3. Print and interpret the results
    # ---------------------------------------------------------
    alpha = 0.05  # Standard significance level

    print("=== Real-Time Factor (RTF) Comparison ===")
    print(f"Test Statistic: {wer_stat:.4f}, p-value: {wer_p_value:.4f}")
    if wer_p_value < alpha:
        print("Result: Significant difference in RTF between the two models.\n")
    else:
        print("Result: No significant difference in RTF.\n")

    print("=== Throughput Per Second (TPS) Comparison ===")
    print(f"Test Statistic: {cer_stat:.4f}, p-value: {cer_p_value:.4f}")
    if cer_p_value < alpha:
        print("Result: Significant difference in TPS between the two models.")
    else:
        print("Result: No significant difference in TPS.")

In [ ]:
wilcoxon_t_test(df_int8=processed_coral_int8_df, df_baseline=processed_coral_baseline_df)

In [ ]:
wer_diff = processed_coral_int8_df['wer'] - processed_coral_baseline_df['wer']
cer_diff = processed_coral_int8_df['cer'] - processed_coral_baseline_df['cer']

avg_wer = wer_diff.mean()
avg_cer = cer_diff.mean()

avg_wer_int8 = processed_coral_int8_df['wer'].mean()
avg_wer_baseline = processed_coral_baseline_df['wer'].mean()

avg_cer_int8 = processed_coral_int8_df['cer'].mean()
avg_cer_baseline = processed_coral_baseline_df['cer'].mean()

wer_pct = ((avg_wer_int8 - avg_wer_baseline) / avg_wer_baseline) * 100
cer_pct = ((avg_cer_int8 - avg_cer_baseline) / avg_cer_baseline) * 100

if wer_pct > 0:
    print(f'The INT8 models WER is {abs(wer_pct):2f}% worse than full precision')
else:
    print(f'The INT8 models WER is {abs(wer_pct):2f}% better than full precision')

if cer_pct > 0:
    print(f'The INT8 models CER is {abs(cer_pct):2f}% worse than full precision')
else:
    print(f'The INT8 models CER is {abs(cer_pct):2f}% better than full precision')


## Experiment 3 - Comparing the same Speaker Diarization model with minimal vs. full audio context
This will essentially compare two different architectural design choices for how to proces audio for speaker diarization by looking at the potential increase or decrease in inference speed, and sample througput per second. As well as an evaluation on whether that had significant impact (negative or positive) on the DER.

In [ ]:
EPOCHS = 2

### Average for full context audio

In [ ]:
import pandas as pd

with open('../results/wa_diarization_performance.log', 'r') as f:
    timings = []
    mem = []
    batch = []
    pattern_time = re.compile(
            r"Walltime:\s*([0-9]+(?:\.[0-9]+)?)\s+CPU time:\s*([0-9]+(?:\.[0-9]+)?)"
        )
    pattern_ram = re.compile(
        r"RSS:\s*([0-9]+(?:\.[0-9]+)?)"
    )
    pattern_batch = re.compile(
        r"Epoch:\s*(\d+),\s*Batch:\s*(\d+)"
    )
    for line_number, line in enumerate(f, start=1):
        m = pattern_time.search(line)
        r = pattern_ram.search(line)
        b = pattern_batch.search(line)
        if r: 
            mem.append({
                "ram": float(r.group(1))
            })
        if m:
            timings.append({
                "walltime": float(m.group(1)),
                "cputime": float(m.group(2))
            })
        if b:
            batch.append({
                'epoch': int(b.group(1)),
                'batch': int(b.group(2))
            })
print(len(timings))
print(len(mem))
print(len(batch))

combined = [{
    "ram": mem["ram"],
    "walltime": timing["walltime"],
    "cputime": timing["cputime"]
} for mem, timing in zip(mem, timings)]

sd_baseline_df = pd.DataFrame.from_dict(combined)
print(sd_baseline_df.shape)
sd_baseline_df.head()

In [ ]:
total_duration = total_duration = 9595.560000000001 * EPOCHS
total_walltime = sd_baseline_df['walltime'].sum()
total_cputime = sd_baseline_df['cputime'].sum()
avg_ram = sd_baseline_df['ram'].mean()

rtf_global = total_walltime / total_duration
tps_global = 1 / rtf_global

print("=== Full Dataset Performance Summary ===")
print(f"Total Audio Duration: {total_duration:.2f} seconds")
print(f"Total Walltime:       {total_walltime:.2f} seconds")
print(f"Total CPU Time:       {total_cputime:.2f} seconds")
print(f"Global Average RAM:   {avg_ram:.2f} MB") # adjust unit as needed
print(f"--------------------------------------")
print(f"True Global RTF:      {rtf_global:.4f}")
print(f"True Global TPS:      {tps_global:.2f}")

### Average for streaming context

In [ ]:
import pandas as pd

with open('../results/wa_streaming_diarization_performance.log', 'r') as f:
    timings = []
    mem = []
    pattern_time = re.compile(
            r"Walltime:\s*([0-9]+(?:\.[0-9]+)?)\s+CPU time:\s*([0-9]+(?:\.[0-9]+)?)"
        )
    pattern_ram = re.compile(
        r"RSS:\s*([0-9]+(?:\.[0-9]+)?)"
    )
    for line_number, line in enumerate(f, start=1):
        m = pattern_time.search(line)
        r = pattern_ram.search(line)
        if r: 
            mem.append({
                "ram": float(r.group(1))
            })
        if m:
            timings.append({
                "walltime": float(m.group(1)),
                "cputime": float(m.group(2))
            })


combined = [{
    "ram": mem["ram"],
    "walltime": timing["walltime"],
    "cputime": timing["cputime"]
} for mem, timing in zip(mem, timings)]

sd_baseline_df = pd.DataFrame.from_dict(combined)
print(sd_baseline_df.shape)
sd_baseline_df.head()

In [ ]:
total_duration = 9595.560000000001 * EPOCHS
total_walltime = sd_baseline_df['walltime'].sum()
total_cputime = sd_baseline_df['cputime'].sum()
avg_ram = sd_baseline_df['ram'].mean()

rtf_global = total_walltime / total_duration
tps_global = 1 / rtf_global

print("=== Full Dataset Performance Summary ===")
print(f"Total Audio Duration: {total_duration:.2f} seconds")
print(f"Total Walltime:       {total_walltime:.2f} seconds")
print(f"Total CPU Time:       {total_cputime:.2f} seconds")
print(f"Global Average RAM:   {avg_ram:.2f} MB") # adjust unit as needed
print(f"--------------------------------------")
print(f"True Global RTF:      {rtf_global:.4f}")
print(f"True Global TPS:      {tps_global:.2f}")

In [ ]:
metadata = pd.read_csv('../data/wrist_angel_metadata.csv')
metadata['duration'] = metadata['end'] - metadata['start']
print(metadata['duration'].sum())

audio_start_map = metadata.set_index('audio_id')['start'].to_dict()


In [ ]:

import json

current_offset = 0.0
current_audio_id = None

streaming_df = []

with open('../results/diarize_wa_streaming.jsonl', 'r') as infile:
    for line in infile:
        segment = json.loads(line)
        
        # 1. Check if we have moved to a new audio file
        audio_id = segment.get('audio_id')
        if audio_id != current_audio_id:
            current_offset = 0.0  # Reset the timeline for the new file
            current_audio_id = audio_id
        
        # 2. Calculate the absolute times using the running offset
        seg_start = current_offset + segment['speaker_start']
        seg_end = current_offset + segment['speaker_end']
        
        # 3. Add this segment's duration to the offset for the next iteration
        segment_duration = segment['speaker_end'] - segment['speaker_start']
        current_offset += segment_duration
        
        streaming_df.append({
            'audio_id': segment['audio_id'],
            'segment_id': segment['segment_id'],
            'start': seg_start,
            'end': seg_end,
            'speaker': segment['speaker'],
            'seg_duration': segment_duration
        })

In [ ]:
import json

def convert_jsonl_to_rttm_by_segment(input_jsonl, output_rttm):
    # Dictionary to track the state of each audio_id to safely handle interleaved files
    audio_states = {}
    
    with open(input_jsonl, 'r') as infile, open(output_rttm, 'w') as outfile:
        for line in infile:
            if not line.strip():
                continue
                
            data = json.loads(line)
            audio_id = data['audio_id']
            seg_id = data['segment_id']
            rel_start = data['speaker_start']
            rel_end = data['speaker_end']
            speaker = data['speaker']
            duration = data['speech_duration']
            
            # 1. Initialize state if this is the first time we see this audio file
            if audio_id not in audio_states:
                
                audio_states[audio_id] = {
                    'offset': 0.0,
                    'current_segment_id': seg_id,
                    'max_end_in_segment': 0.0
                }
            
            state = audio_states[audio_id]
            
            # 2. Check if we moved to a NEW segment for this audio file
            if seg_id != state['current_segment_id']:
                # Advance the timeline offset by the length of the previous segment 
                # (based on the last time someone spoke in it)
                state['offset'] += state['max_end_in_segment']
                
                # Update the tracker to the new segment and reset the max end
                state['current_segment_id'] = seg_id
                state['max_end_in_segment'] = 0.0 
            
            # 3. Calculate absolute start time based on the current offset
            offset = audio_start_map.get(audio_id)
            abs_start = state['offset'] + rel_start + offset
            
            # 4. Update the maximum end time seen so far in this current segment
            if rel_end > state['max_end_in_segment']:
                state['max_end_in_segment'] = rel_end
            
            # 5. Format as standard RTTM and write to file
            rttm_line = f"SPEAKER {audio_id} 1 {abs_start:.3f} {duration:.3f} <NA> <NA> {speaker} <NA> <NA>\n"
            outfile.write(rttm_line)

# Run the conversion
convert_jsonl_to_rttm_by_segment('../results/diarize_wa_streaming.jsonl', '../results/wa_streaming.rttm')

In [ ]:
# First convert the diarization output to the right rttm format...
rttm_results_path = '../results/diarization_baseline_wa.rttm'

with open('../results/diarize_wa_baseline.jsonl', 'r') as reader, open(rttm_results_path, 'w') as writer:
    for line in reader:
        item = json.loads(line)
        file_id = item['audio_id']
        start = item['speaker_start']
        offset = audio_start_map.get(file_id)
        start = start + offset
        duration = item['speech_duration']
        label = item['speaker']
        
        record = f'SPEAKER {file_id} 1 {start} {duration} <NA> <NA> {label} <NA> <NA>'
        writer.write(record + '\n')
        writer.flush()

In [ ]:
# First convert the diarization output to the right rttm format...
rttm_results_path = '../results/diarization_baseline_wa.rttm'

with open('../results/diarize_wa_baseline.jsonl', 'r') as reader, open(rttm_results_path, 'w') as writer:
    for line in reader:
        item = json.loads(line)
        file_id = item['audio_id']
        start = item['speaker_start']
        offset = audio_start_map.get(file_id)
        start = start + offset
        duration = item['speech_duration']
        label = item['speaker']
        
        record = f'SPEAKER {file_id} 1 {start} {duration} <NA> <NA> {label} <NA> <NA>'
        writer.write(record + '\n')
        writer.flush()

In [ ]:
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

def load_annotations_from_rttm(filepath):
    """
    Reads an RTTM file and returns a dictionary of pyannote Annotation objects.
    
    Standard RTTM format:
    SPEAKER <audio_id> <channel> <start> <duration> <NA> <NA> <speaker_id> <NA> <NA>
    """
    annotations = {}
    
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            
            # Skip empty lines or non-speaker lines
            if not parts or parts[0] != "SPEAKER":
                continue
                
            audio_id = parts[1]
            start_time = float(parts[3])
            duration = float(parts[4])
            speaker_id = parts[7]
            
            # Calculate absolute end time
            end_time = start_time + duration
            
            # Initialize the Annotation object if this is a new audio file
            if audio_id not in annotations:
                annotations[audio_id] = Annotation(uri=audio_id)
            
            # Add the segment to the pyannote Annotation object
            annotations[audio_id][Segment(start_time, end_time)] = speaker_id
            
    return annotations


In [ ]:
print("Loading Reference Annotations...")
reference_dict = load_annotations_from_rttm('../results/ground_truth/transcripts.rttm')
print("Loading Hypothesis Annotations...")
hypothesis_dict = load_annotations_from_rttm('../results/wa_streaming.rttm')

metric = DiarizationErrorRate()

print("\n--- Per-File Results ---")

for audio_id, ref_annotation in reference_dict.items():
    hyp_annotation = hypothesis_dict.get(audio_id, Annotation(uri=audio_id))
    file_der = metric(ref_annotation, hyp_annotation)
    print(f"{audio_id}: {file_der * 100:.2f}%")
global_der = abs(metric)
print(f"\n=== Final Dataset-Wide DER ===: {global_der * 100:.2f}%")
print("\n--- Detailed Metric Report ---")
report_df_streaming = metric.report(display=True)

In [ ]:
print("Loading Reference Annotations...")
reference_dict = load_annotations_from_rttm('../results/ground_truth/transcripts.rttm')
print("Loading Hypothesis Annotations...")
hypothesis_dict = load_annotations_from_rttm('../results/diarization_baseline_wa.rttm')

metric = DiarizationErrorRate()
print("\n--- Per-File Results ---")
for audio_id, ref_annotation in reference_dict.items():
    hyp_annotation = hypothesis_dict.get(audio_id, Annotation(uri=audio_id))
    file_der = metric(ref_annotation, hyp_annotation)
    print(f"{audio_id}: {file_der * 100:.2f}%")

global_der = abs(metric)
print(f"\n=== Final Dataset-Wide DER ===: {global_der * 100:.2f}%")
print("\n--- Detailed Metric Report ---")
report_df_baseline = metric.report(display=True)

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

der_column = ('diarization error rate', '%')

streaming_der = report_df_streaming.drop(index='TOTAL', errors='ignore')[der_column]
non_streaming_der = report_df_baseline.drop(index='TOTAL', errors='ignore')[der_column]

df_paired = pd.concat([streaming_der, non_streaming_der], axis=1, keys=['streaming', 'non_streaming']).dropna()
statistic, p_value = wilcoxon(df_paired['streaming'], df_paired['non_streaming'], alternative='two-sided')

print("=== Wilcoxon Signed-Rank Test Results ===")
print(f"Test Statistic: {statistic}")
print(f"P-value:        {p_value:.5f}")
print("-----------------------------------------")

alpha = 0.05

if p_value < alpha:
    print("Conclusion: There is a STATISTICALLY SIGNIFICANT difference between the approaches.")
    
    mean_diff = streaming_der.mean() - non_streaming_der.mean()
    if mean_diff > 0:
        print(f"-> Non-streaming performed significantly better (DER was lower by {abs(mean_diff):.4f}).")
    else:
        print(f"-> Streaming performed significantly better (DER was lower by {abs(mean_diff):.4f}).")
else:
    print("Conclusion: NO statistically significant difference was found between the approaches.")

In [ ]:
print(df_paired.columns)